In [89]:
"""
IMPORT PACKAGES
"""
from datetime import date, timedelta, datetime
from massive import RESTClient
import polars as pl
import duckdb
from extract_orats import ExtractOratsEngine

---
---
---

In [2]:
"""
INPUT VARIABLES THAT I REQUIRE

EQUITY DATA: retrieved from MASSIVE
"""

MASSIVE_API_KEY="JqmhWpb1vmo5kjpneuy7ssu3rJ0tS6Mu"
tickers = ["AAPL"]
price_column = ["close", "volume"]
time_column = "time"
max_age_days = 60 # default by Mislav
min_history_days = 252 # default by Mislav
max_history_days = 50000 # using this to get all data available from massive
lookback_days = 22

In [3]:
"""
INITIAL SET UP
"""

today = date.today().strftime("%Y-%m-%d")
previous = max(date(1970, 1, 1), date.today() - timedelta(days=max_history_days)).strftime("%Y-%m-%d")

print(f"Today: {today}")
print(f"Previous: {previous}")

client = RESTClient(MASSIVE_API_KEY)

Today: 2026-03-10
Previous: 1970-01-01


# EQUITY DATA
---

In [4]:
def map_massive(columns: str | list):
    """
    map the columns to massive
    """
    map_columns = {
        "close": "c",
        "high": "h",
        "low": "l",
        "open": "o",
        "volume": "v",
        "time": "t"
    }
    if isinstance(columns, list):
        return [map_columns[col] for col in columns]
    return map_columns[columns]

In [5]:
def get_ticker_df(ticker: str) -> tuple[pl.DataFrame, pl.DataFrame]:
    """
    Extracts Equity Data for One Single Ticker

    Input: ticker : string (ONE TICKER)
    Output:
        (1) df : time, ticker, close, volume
        (2) latest: ticker, time, close - 1 ROW ONLY
        (3) liq: ticker, time, vol_22d, dol_vol_22d

    """
    aggs = []
    for a in client.list_aggs(
        ticker,
        multiplier=1,
        timespan="day",
        from_=previous,
        to=today,
        adjusted="true",
        sort="asc",
        limit=50000,
    ):
        aggs.append(a)

    agg_attr = {
        "close": "close",
        "high": "high",
        "low": "low",
        "open": "open",
        "volume": "volume",
        "time": "timestamp",
    }

    columns = price_column + [time_column]

    df = pl.DataFrame([
        {col: getattr(a, agg_attr[col]) for col in columns}
        for a in aggs
    ])

    # keep time as Date type for correct sorting, format for display at the end
    df = df.with_columns(
        pl.from_epoch("time", time_unit="ms").cast(pl.Date),
        pl.lit(ticker).alias("ticker"),
    ).select(["time", "ticker"] + price_column)

    # latest price
    latest = df.sort("time").tail(1).select(["ticker", "time", "close"])

    # liquidity: sum volume and dollar volume over last N days
    liq = (
        df.sort("time")
        .tail(lookback_days)
        .select(
            pl.col("time").last().alias("time"),
            pl.lit(ticker).alias("ticker"),
            pl.col("volume").sum().alias("vol_22d"),
            (pl.col("close") * pl.col("volume")).sum().alias("dollar_vol_22d"),
        )
        .head(1)
    )

    # format time as dd-mm-yyyy for display
    # df = df.with_columns(pl.col("time").str.strptime(pl.Date, "%d-%m-%Y"))

    return df, latest, liq

In [56]:
dfs, latests, liqs = [], [], []
for ticker in tickers:
    df, latest, liq = get_ticker_df(ticker)
    dfs.append(df)
    latests.append(latest)
    liqs.append(liq)

equity_df = pl.concat(dfs)
latest = pl.concat(latests) # latest close price
liq = pl.concat(liqs) # latest volume

---

# OPTIONS DATA
---

1. Go look at ORATS data (from implied_volatility_term_structure)

What he did here is basically a full pipeline of what he has there

OR test4.ipynb which is the whole thing, but it is not unified

Understand and Run term_structure_volatiltiy.py (section 1-3) and trade_slope.py (section 4)

In [125]:
def update_options_duckdb():
    """
    refresh the duckdb
    """
    import duckdb
    from extract_orats import ExtractOratsEngine

    with ExtractOratsEngine() as engine:
        stats = engine.sync_required_years_sequential(verbose=True)

In [126]:
update_options_duckdb()

✓ FTP Login successful: 230 User logged in, proceed.
✓ Connected to database: data/ORATS_OPTIONS_DB.duckdb
ORATS Data Sync (Required Years Only): 2007 - 2026

Skipping complete years:
  2007: 251 dates exist, 1 trading days missing, 114 total calendar gaps (OK)
  2008: 253 trading dates present, 113 calendar gaps are weekends/holidays (OK)
  2009: 252 trading dates present, 113 calendar gaps are weekends/holidays (OK)
  2010: 252 trading dates present, 113 calendar gaps are weekends/holidays (OK)
  2011: 252 trading dates present, 113 calendar gaps are weekends/holidays (OK)
  2012: 250 dates exist, 2 trading days missing, 116 total calendar gaps (OK)
  2013: 252 trading dates present, 113 calendar gaps are weekends/holidays (OK)
  2014: 252 trading dates present, 113 calendar gaps are weekends/holidays (OK)
  2015: 252 trading dates present, 113 calendar gaps are weekends/holidays (OK)
  2016: 252 trading dates present, 114 calendar gaps are weekends/holidays (OK)
  2017: 251 trading 

Syncing 2026: 100%|██████████| 1/1 [01:00<00:00, 60.95s/it]



✓ Appended 904,512 rows to 'options_data_2026'
  Files extracted: 1, Skipped (no data): 0

SYNC COMPLETE
Years updated: 1
Total rows added: 904,512
✓ FTP connection closed
✓ DuckDB connection closed


In [8]:
from datetime import date

def extract_relevant_options(tickers: list):
    """
    Extracts relevant options based on:
    1. Specific Desired Tickers
    2. Options that have expired dates > today's date
    """
    ticker_list = ", ".join(f"'{t}'" for t in tickers)
    year = date.today().year
    tables = [f"options_data_{y}" for y in [year - 1, year]]

    con = duckdb.connect('data/ORATS_OPTIONS_DB.duckdb', read_only=True)

    union_query = " UNION ALL ".join(
        f"SELECT * FROM {t}" for t in tables
    )

    df = pl.from_pandas(con.execute(f"""
        SELECT * FROM ({union_query})
        WHERE expirDate >= CURRENT_DATE
        AND ticker IN ({ticker_list})
    """).fetchdf())

    con.close()
    return df


In [127]:
options_df = extract_relevant_options(tickers=tickers)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

---

# BEGIN SCREENING

---

In [10]:
equity_df

time,ticker,close,volume
date,str,f64,f64
2003-09-10,"""AAPL""",0.3961,2.24893256e8
2003-09-11,"""AAPL""",0.4029,2.13937248e8
2003-09-12,"""AAPL""",0.4125,1.80005112e8
2003-09-15,"""AAPL""",0.3973,2.26843064e8
2003-09-16,"""AAPL""",0.3993,2.69379264e8
…,…,…,…
2026-03-03,"""AAPL""",263.75,3.8569e7
2026-03-04,"""AAPL""",262.52,3.9803e7
2026-03-05,"""AAPL""",260.29,4.9659e7


In [128]:
options_df

ticker,expirDate,trade_date,cOpra,pOpra,stkPx,yte,strike,cVolu,cOi,pVolu,pOi,cBidPx,cValue,cAskPx,pBidPx,pValue,pAskPx,cBidIv,cMidIv,cAskIv,smoothSmvVol,pBidIv,pMidIv,pAskIv,iRate,divRate,residualRateData,delta,gamma,theta,vega,rho,phi,driftlessTheta,extVol,extCTheo,extPTheo,spot_px
str,datetime[μs],datetime[μs],str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""AAPL""",2026-06-18 00:00:00,2025-01-02 00:00:00,"""AAPL260618C00005000""","""AAPL260618P00005000""",243.55,1.45753,5.0,0.0,31.0,0.0,11.0,236.7,238.55,239.9,0.0,0.0,0.02,0.0,1.14907,2.298143,0.303182,0.0,0.56518,1.130366,0.0425,0.0,0.0,1.0,0.0,0.0,0.00001,0.0,0.0,0.0,0.352054,238.550003,7.7631e-25,0.0
"""AAPL""",2026-06-18 00:00:00,2025-01-02 00:00:00,"""AAPL260618C00010000""","""AAPL260618P00010000""",243.55,1.45753,10.0,0.0,1.0,1.0,4.0,231.75,233.55,234.9,0.01,0.0,0.02,0.0,0.83632,1.672636,0.303182,0.862957,0.88914,0.915318,0.0425,0.0,-0.002824,1.0,0.0,0.0,0.00001,0.0,0.0,0.0,0.352054,233.550003,6.9828e-16,0.0
"""AAPL""",2026-06-18 00:00:00,2025-01-02 00:00:00,"""AAPL260618C00015000""","""AAPL260618P00015000""",243.55,1.45753,15.0,0.0,0.0,0.0,0.0,226.85,228.55,229.95,0.0,0.01,0.08,0.0,0.69641,1.392819,0.303182,0.0,0.4528,0.905609,0.0425,0.0,0.0,1.0,0.0,0.0,0.00001,0.0,0.0,0.0,0.352054,228.550003,5.6206e-12,0.0
"""AAPL""",2026-06-18 00:00:00,2025-01-02 00:00:00,"""AAPL260618C00020000""","""AAPL260618P00020000""",243.55,1.45753,20.0,0.0,1.0,0.0,12.0,222.1,223.55,225.1,0.0,0.0,0.07,0.0,0.61252,1.225046,0.303182,0.0,0.39905,0.798099,0.0425,0.0,0.0,0.999847,0.000034,0.0,0.00001,0.0,0.0,0.0,0.352054,223.568909,1.0979e-9,0.0
"""AAPL""",2026-06-18 00:00:00,2025-01-02 00:00:00,"""AAPL260618C00025000""","""AAPL260618P00025000""",243.55,1.45753,25.0,0.0,0.0,0.0,0.0,217.45,218.72,220.35,0.01,0.0,0.11,0.0,0.5519,1.103803,0.303182,0.611106,0.68702,0.762934,0.0425,0.0,-0.002841,0.996118,0.000038,-0.000373,0.003905,0.240572,-2.197192,-0.00011,0.352054,218.737717,4.8135e-8,0.0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""AAPL""",2028-12-15 00:00:00,2026-03-09 00:00:00,"""AAPL281215C00480000""","""AAPL281215P00480000""",260.06,2.7726,480.0,0.0,86.0,0.0,0.0,7.05,7.44,7.75,217.5,219.94,222.5,0.249434,0.25279,0.25614,0.251847,0.0,0.19342,0.386847,0.0359,0.0,-0.003276,0.152349,0.002157,-0.015792,1.046513,0.895957,-1.09846,-0.012568,0.282536,10.689508,219.940002,0.0
"""AAPL""",2028-12-15 00:00:00,2026-03-09 00:00:00,"""AAPL281215C00490000""","""AAPL281215P00490000""",260.06,2.7726,490.0,0.0,39.0,0.0,0.0,6.4,6.74,7.15,227.5,229.94,232.5,0.249149,0.25307,0.256989,0.251376,0.0,0.19807,0.396142,0.0359,0.0,-0.003288,0.140377,0.002046,-0.014863,0.921292,0.828977,-1.012139,-0.011875,0.282225,9.889568,229.940002,0.0
"""AAPL""",2028-12-15 00:00:00,2026-03-09 00:00:00,"""AAPL281215C00500000""","""AAPL281215P00500000""",260.06,2.7726,500.0,5.0,411.0,0.0,0.0,5.75,6.13,6.45,237.5,239.94,242.5,0.248435,0.25216,0.255886,0.25085,0.0,0.20262,0.405243,0.0359,0.0,-0.003301,0.129305,0.001936,-0.013961,0.942922,0.76654,-0.93231,-0.011194,0.281972,9.09499,239.940002,0.0


In [18]:
def run_pipeline(options_df:pl.DataFrame, equity_df:pl.DataFrame):
    # TERM STRUCTURE ANALYSIS — filter, compute IV term structure, rank by SLOPE
    from term_structure_volatility import TermStructureVolatilityEngine

    engine = TermStructureVolatilityEngine()

    # 1. Cast dates and select columns needed by the engine
    opts = options_df.with_columns([
        pl.col("trade_date").cast(pl.Date),
        pl.col("expirDate").cast(pl.Date),
    ]).select([
        "ticker", "trade_date", "expirDate", "yte", "strike", "stkPx",
        "cBidPx", "cAskPx", "cMidIv", "pBidPx", "pAskPx", "pMidIv",
        "cBidIv", "pBidIv", "delta"
    ])

    # 2. Apply 5 filters from Campasano & Linn paper
    opts = engine.filter_options(opts)
    print(f"After filtering: {opts.shape}")

    # 3. ATM options → short-term IV (IV1M, ~30d) and long-term IV (IVLT, >=180d)
    atm_options = engine.identify_atm_options(opts)
    short_term_iv = engine.compute_short_term_iv(atm_options)
    long_term_iv = engine.compute_long_term_iv(atm_options)

    # Rename columns to avoid collisions, then merge
    short_cols = [c for c in short_term_iv.columns if c not in ["ticker", "trade_date"]]
    short_term_iv = short_term_iv.rename({c: f"{c}_short" for c in short_cols})
    long_cols = [c for c in long_term_iv.columns if c not in ["ticker", "trade_date"]]
    long_term_iv = long_term_iv.rename({c: f"{c}_long" for c in long_cols})

    iv_data = short_term_iv.join(long_term_iv, on=["ticker", "trade_date"], how="inner")
    iv_data = iv_data.select([
        "ticker", "trade_date", "stkPx_short",
        pl.col("ATM_IV_short").alias("IV1M"),
        pl.col("ATM_IV_long").alias("IVLT"),
        pl.col("days_to_expiry_short").alias("days_to_expiry"),
        pl.col("cBidPx_short").alias("cBidPx"), pl.col("cAskPx_short").alias("cAskPx"),
        pl.col("pBidPx_short").alias("pBidPx"), pl.col("pAskPx_short").alias("pAskPx"),
    ])

    # 4. SLOPE = (IVLT - IV1M) / IVLT
    iv_data = engine.compute_slope(iv_data)
    print(f"IV term structure: {iv_data.shape}")

    # 5. RVLT (252-day realized vol) from equity data
    eq = (
        equity_df.sort("time")
        .with_columns((pl.col("close") / pl.col("close").shift(1)).log().alias("returns"))
        .with_columns(pl.col("returns").pow(2).rolling_sum(window_size=252).sqrt().alias("RVLT"))
        .drop_nulls(subset=["RVLT"])
        .select("ticker", pl.col("time").alias("trade_date"), "close", "RVLT")
    )

    # 6. Merge equity + options, compute IVRV_SLOPE and straddle prices
    iv_data = iv_data.join(eq, on=["ticker", "trade_date"], how="left")
    iv_data = engine.compute_ivrv_slope(iv_data)
    iv_data = engine.compute_straddle_prices(iv_data)
    iv_data = iv_data.drop_nulls()

    # 7. Rank SLOPE into quantile tiles (across all dates for single-ticker)
    n_tiles = 10  # deciles — change to 5 for quintiles, 20 for vigintiles
    iv_data = iv_data.with_columns(
        ((pl.col("SLOPE").rank("ordinal") - 1) * n_tiles // pl.col("SLOPE").count() + 1)
        .cast(pl.Int32).alias("q")
    )

    print(f"Final ranked: {iv_data.shape}")
    # iv_data.select(["ticker", "trade_date", "IV1M", "IVLT", "SLOPE", "RVLT", "IVRV_SLOPE", "straddle_price_mid", "q"])
    return(iv_data)

In [20]:
# example usage
# run_pipeline(options_df=options_df, equity_df=equity_df)

---
---
---
---
---

In [28]:
options_df.shape

(175781, 39)

# OWN DOING

In [59]:
# options
FILTER_CONFIG = {
    "min_stock_price": 10,
    "min_delta": 0.35,
    "max_delta": 0.65,
    "min_iv": 0.03, # CHANGED
    "max_iv": 2.0,
    "short_term_max_days": 60,
    "short_term_target_days": 30,
    "long_term_min_days": 180,
}

# equity
FILTER_CONFIG2 = {
    "min_price": 10.0,
    "max_age_days": 60,
    "min_volume_22d": 300_000,
    "min_dollar_volume": 100_000_000,
    "min_history_days": 252,
    "max_abs_daily_return": 0.5,
    "lookback_days": 22,
    "top_n_dollar_volume": None,
}



cfg = FILTER_CONFIG
cfg2 = FILTER_CONFIG2

In [169]:
# all 5 filters in the paper
# Filter 1: stkPx > $10
# Filter 2: bid-ask spread > 0 (no stale quotes)
# Filter 3: bid prices > 0 (liquid options)
# Filter 4: |delta| between 0.35 and 0.65 (ATM range)
# Filter 5: IV between 3% and 200%

filtered_options_df = options_df.filter(
    (pl.col("stkPx") > cfg["min_stock_price"]) &
    ((pl.col("cBidPx") - pl.col("cAskPx")).abs() > 0) &
    ((pl.col("pBidPx") - pl.col("pAskPx")).abs() > 0) &
    (pl.col("cBidPx") > 0) & 
    (pl.col("pBidPx") > 0) &
    (pl.col("delta").abs() > cfg["min_delta"]) &
    (pl.col("delta").abs() < cfg["max_delta"]) &
    (pl.col("cBidIv") > cfg["min_iv"]) &
    (pl.col("cBidIv") < cfg["max_iv"]) &
    (pl.col("pBidIv") > cfg["min_iv"]) &
    (pl.col("pBidIv") < cfg["max_iv"])
)

In [151]:
# add rolling 22-day volume and dollar volume per ticker
# add rolling 22-day absolute daily return and maximum absolute return
# add rolling 252-day RVLT
equity_df = (equity_df.sort(["ticker", "time"]).with_columns(
    (pl.col("close") / pl.col("close").shift(1) - 1).over("ticker").alias("abs_daily_return").abs(),
    (pl.col("close") / pl.col("close").shift(1) - 1).over("ticker").alias("returns"),
    pl.col("volume").rolling_sum(window_size=22).over("ticker").alias("vol_22d"),
    (pl.col("close") * pl.col("volume")).rolling_sum(window_size=22).over("ticker").alias("dollar_vol_22d"),)
    .with_columns(
        pl.col("abs_daily_return").rolling_max(window_size=22).over("ticker").alias("max_abs_ret"),
        (pl.col("returns").pow(2).rolling_sum(window_size=252).sqrt()).alias("RVLT"),
    ).drop_nulls())


In [69]:
cutoff = date.today() - timedelta(days=cfg2["max_age_days"])

In [153]:
# filtering equity_df based on mislav's code (ignoring filtering by dates)
# source: https://github.com/daebeken/options-scanner-duplicate/blob/main/filter_symbols.py#L120-L125

# interpretation: if the last trading day appears in the last row, it means that the ticker is able to trade

filtered_equity_df = equity_df.filter(
    (pl.col("close") >= cfg2["min_price"]) &
    (pl.col("vol_22d") >= cfg2["min_volume_22d"]) &
    (pl.col("dollar_vol_22d") >= cfg2["min_dollar_volume"]) &
    (pl.col("max_abs_ret") <= cfg2["max_abs_daily_return"])
)

In [170]:
# adding a few additional columns to options

# (1) finding the difference in stock and option price
# (2) finding the ATM implied volatility
# (3) finding the days to expiry

filtered_options_df = filtered_options_df.with_columns(
    ((pl.col("strike") - pl.col("stkPx")).abs()).alias("atm_diff"),
    ((pl.col("cMidIv") + pl.col("pMidIv")) / 2).alias("atm_iv"),
    (pl.col("expirDate").cast(pl.Date) - pl.lit(datetime.now().date())).dt.total_days().alias("days_to_expiry"),
)


In [171]:
# 1 ROW, 1 TRADE DATE, NEAREST EXPIRY OPTION

# cm_df provides a daily time series of constant-maturity implied volatility per ticker. 
# For each trade date, it interpolates across all available expiries' ATM IVs to produce standardized IV30D (30-day) 
# and IV180D (180-day) values. This removes the effect of time decay from the IV term structure, making it possible to 
# compare implied volatility levels consistently across different trade dates.

# LOGIC CHAIN: COMPUTING THE CONSTANT MATURITY using the option contract that has the CLOSEST STRIKE PRICE TO CURRENT STOCK PRICE

# (1) Filter to near-ATM options (delta 0.35–0.65)
# (2) Pick the closest strike to stock price per expiry → one IV per expiry
# (3) Interpolate across expiries to get constant-maturity IV

# Returns: the shortest contract (days to expiry is the shortest) one

import numpy as np

# Step 1: ATM option per ticker/trade_date/expiry (smallest atm_diff): pick the closest strike price to the current stock price
atm_df = (filtered_options_df.sort("atm_diff").group_by(["ticker", "trade_date", "expirDate"]).first())

# Step 2: Interpolate to constant maturity tenors
tenors = [30, 180]

def interpolate_group(group):
    x = group["days_to_expiry"].to_numpy().astype(float)
    y = group["atm_iv"].to_numpy().astype(float)

    mask = np.isfinite(x) & np.isfinite(y)
    x, y = x[mask], y[mask]

    if len(x) == 0:
        return {f"IV{t}D": np.nan for t in tenors}
    if len(np.unique(x)) < 2:
        return {f"IV{t}D": float(y[0]) for t in tenors}

    ux, idx = np.unique(x, return_inverse=True)
    uy = np.array([y[idx == i].mean() for i in range(len(ux))])
    return {f"IV{t}D": float(np.interp(t, ux, uy)) for t in tenors}

results = []

# SHOWING SHORTEST CONTRACT ONE
for (ticker, trade_date), group in atm_df.group_by(["ticker", "trade_date"]):
    group = group.sort("days_to_expiry")
    ivs = interpolate_group(group)
    results.append({"ticker": ticker,"expirDate": group["expirDate"][0] ,"trade_date": trade_date, "cOpra": group["cOpra"][0], "pOpra": group["pOpra"][0], "stkPx": group["stkPx"][0], **ivs})

cm_df = pl.DataFrame(results)

In [172]:
filtered_options_df2 = filtered_options_df.join(cm_df, on=["ticker", "expirDate", "trade_date", "cOpra", "pOpra"], how="inner")

# SELECTION OF OPTIONS DATA AND EQUITY DATA

In [174]:
filtered_options_df3 = filtered_options_df2.select(["ticker", "expirDate", "trade_date", "cOpra", "pOpra", "cBidPx", "cAskPx", "pBidPx", "pAskPx","IV30D", "IV180D"])

In [186]:
equity_df2 = equity_df.select(["time", "ticker", "returns", "dollar_vol_22d", "RVLT"]).rename({"time": "trade_date", "dollar_vol_22d":"dollar_vol"})

In [188]:
consolidated_df = filtered_options_df3.with_columns(pl.col("trade_date").cast(pl.Date)).join(equity_df2,
    on=["ticker", "trade_date"], how="inner")

In [189]:
# every row represents the option contract whose strike price is the closest to the stock price on trade_date
# also, we are looking at ACTIVE option contracts (those that have not expired)
consolidated_df = consolidated_df.with_columns(
    ((pl.col("IV180D") - pl.col("IV30D")) / pl.col("IV180D")).alias("SLOPE"),
    ((pl.col("RVLT") - pl.col("IV30D")) / pl.col("RVLT")).alias("IVRV_SLOPE"),
    (pl.col("cBidPx") + pl.col("pBidPx")).alias("straddle_price_bid"),
    (pl.col("cAskPx") + pl.col("pAskPx")).alias("straddle_price_ask"),
    ((pl.col("cBidPx") + pl.col("pBidPx") + pl.col("cAskPx") + pl.col("pAskPx")) / 2).alias("straddle_price_mid"),
)

# SECOND ROUND OF SELECTION

In [ ]:
filtered_consolidated_df = consolidated_df.select(
    ["ticker", "expirDate", "trade_date", "cOpra", "pOpra", "dollar_vol", "SLOPE", "IVRV_SLOPE", "straddle_price_bid", "straddle_price_ask", "straddle_price_mid",]
    ).with_columns((pl.col("straddle_price_bid").shift(-1).over("ticker") / pl.col("straddle_price_ask") - 1).alias("wret"),)

In [191]:
filtered_consolidated_df

ticker,expirDate,trade_date,cOpra,pOpra,dollar_vol,SLOPE,IVRV_SLOPE,straddle_price_bid,straddle_price_ask,straddle_price_mid,wret
str,datetime[μs],date,str,str,f64,f64,f64,f64,f64,f64,f64
"""AAPL""",2026-06-18 00:00:00,2025-01-02,"""AAPL260618C00245000""","""AAPL260618P00245000""",2.5768e11,0.000339,-0.083238,57.5,58.55,58.025,-0.035867
"""AAPL""",2026-06-18 00:00:00,2025-01-03,"""AAPL260618C00245000""","""AAPL260618P00245000""",2.5594e11,0.000027,-0.069768,56.45,57.85,57.15,-0.014693
"""AAPL""",2026-06-18 00:00:00,2025-01-06,"""AAPL260618C00245000""","""AAPL260618P00245000""",2.5755e11,0.000054,-0.071829,57.0,57.75,57.375,-0.01039
"""AAPL""",2026-06-18 00:00:00,2025-01-07,"""AAPL260618C00240000""","""AAPL260618P00240000""",2.5666e11,-0.003892,-0.086741,57.15,57.75,57.45,-0.01039
"""AAPL""",2026-06-18 00:00:00,2025-01-08,"""AAPL260618C00240000""","""AAPL260618P00240000""",2.5606e11,-0.005198,-0.090601,57.15,57.65,57.4,-0.017346
…,…,…,…,…,…,…,…,…,…,…,…
"""AAPL""",2026-03-11 00:00:00,2026-03-03,"""AAPL260311C00262500""","""AAPL260311P00262500""",3.1137e11,0.050695,0.153838,9.1,9.4,9.25,-0.159574
"""AAPL""",2026-03-11 00:00:00,2026-03-04,"""AAPL260311C00262500""","""AAPL260311P00262500""",2.9783e11,0.051961,0.146905,7.9,8.1,8.0,-0.061728
"""AAPL""",2026-03-11 00:00:00,2026-03-05,"""AAPL260311C00257500""","""AAPL260311P00257500""",2.9080e11,0.06902,0.143374,7.6,7.8,7.7,-0.076923
